# Scaling Laws
### How loss becomes predictable — straight lines across seven orders of magnitude

---

## 📖 The Story

It is 2020. OpenAI's Jared Kaplan and collaborators are staring at a strange-looking plot. They have trained dozens of small language models — from a few thousand to over a billion parameters — and plotted each one's test loss against its size on a log-log graph. The points fall on a near-perfect straight line, spanning seven orders of magnitude. The same thing happens when they plot loss against dataset size, and against total training compute.

That is not supposed to happen. Deep learning is famously empirical and unpredictable — architectures that work beautifully at one scale often fail at another. But here, a simple power law seems to govern how good a language model gets, almost independent of the exact architecture details. Kaplan's team publishes "Scaling Laws for Neural Language Models," and OpenAI uses the resulting curves to plan GPT-3 before training it — deciding how many parameters and how much data to use, in advance, based on extrapolating a line.

Two years later, DeepMind's Hoffmann and collaborators revisit the question with a more careful experimental design. They discover that Kaplan's team had, in effect, answered the right question slightly wrong: at any fixed compute budget, GPT-3-era models were trained with far too many parameters and far too little data. Their model, **Chinchilla** — 4x smaller than Gopher but trained on 4x more tokens, for the *same* compute budget — beats it decisively. This single correction reshaped how every major lab since (LLaMA, and beyond) decides how big to make a model versus how much data to feed it.

---

**What this notebook covers:**
- Generating synthetic "training runs" with a known ground-truth loss surface — since we cannot afford real ones
- Fitting single-variable power laws by hand (log-log linear regression) — and seeing where a naive approach goes subtly wrong
- Fitting the full two-variable Chinchilla loss surface L(N, D) = E + A/N^α + B/D^β with `scipy.optimize.curve_fit`
- Deriving the compute-optimal allocation N_opt(C), D_opt(C) from the fitted exponents
- Reproducing IsoFLOP curves and placing real models (GPT-3, Gopher, Chinchilla) on the compute-vs-size plane

**Prerequisites:**
- Transformer Architecture (Topic 6) — scaling laws describe how that architecture's *loss* behaves as it grows
- Basic curve fitting / linear regression, logarithms

**Note:** We do not have a GPU budget to train hundreds of real language models, so this notebook uses **synthetic loss data generated from the actual published Chinchilla coefficients** (E=1.69, A=406.4, B=410.7, α=0.34, β=0.28). The question we answer from scratch is: *if you only had the noisy loss numbers from a handful of small, cheap runs, could you recover the law that governs the expensive ones?*

In [ ]:
# --- All Imports ---
import numpy as np                        # Core math — power-law fitting from scratch
import matplotlib.pyplot as plt           # Plotting
import seaborn as sns                     # Heatmaps / styling
from scipy.optimize import curve_fit      # Library — joint nonlinear least squares
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
print('All imports successful ✅')

## Part 1: Theory Recap

Five things to know before we write any code:

- **A scaling law is an empirical power law**: test loss falls off as L(N) ≈ (N_c/N)^α as parameters N grow (holding data/compute non-limiting), and similarly for dataset size D and compute C. On log-log axes this is a straight line.
- **The combined surface has an irreducible floor**: L(N, D) = E + A/N^α + B/D^β. E is the entropy of natural text itself — the loss no model, however large, can beat. A/N^α is the penalty for a finite model; B/D^β is the penalty for finite data.
- **Compute ties N and D together**: training compute is approximately C ≈ 6ND FLOPs. For any fixed C, you can spend it on a bigger model (↑N, ↓D) or more data (↑D, ↓N) — scaling laws tell you the *optimal* split.
- **Kaplan (2020) vs. Chinchilla (2022) disagree on that split**: Kaplan's fits favored growing N much faster than D, which is why GPT-3-era models are huge but trained on comparatively little data. Hoffmann et al.'s more careful fit found N and D should grow at close to the *same* rate.
- **Compute-optimal ≠ inference-optimal**: the Chinchilla formula minimizes *training* loss for a fixed training budget. It says nothing about the cost of serving the model afterward — which is why many production models are deliberately trained on more tokens than the formula recommends.

## Setting Up: Synthetic Training Runs

Real scaling-law papers train dozens to hundreds of models at different (N, D) pairs and record the final loss. We cannot do that here, so we simulate it: we pick a **ground-truth loss surface** using the actual coefficients reported in the Chinchilla paper, generate a grid of (N, D) pairs in a *cheap, plausible* range (millions to billions of parameters — not GPT-3 scale), evaluate the true loss at each point, and add a touch of realistic training noise.

The point of building it this way: we know the right answer in advance, so we can check whether our from-scratch fitting code actually recovers it.

In [ ]:
# Ground-truth coefficients (Hoffmann et al., 2022 — the actual published Chinchilla fit)
E_true, A_true, B_true   = 1.69, 406.4, 410.7      # irreducible loss, param penalty, data penalty
alpha_true, beta_true    = 0.34, 0.28               # param exponent, data exponent

def true_loss(N, D):
    """The Chinchilla parametric loss surface: L(N, D) = E + A/N^alpha + B/D^beta"""
    return E_true + A_true / N**alpha_true + B_true / D**beta_true

# A grid of CHEAP, plausible (N, D) pairs -- like a real scaling-law study would run
Ns = np.array([6e6, 1.3e7, 3.5e7, 9e7, 2.4e8, 6e8, 1.6e9, 4e9])     # 6M -> 4B params
Ds = np.array([1e8, 3e8, 1e9, 3e9, 1e10, 3e10, 1e11, 3e11])         # 100M -> 300B tokens

NN, DD = np.meshgrid(Ns, Ds)
NN, DD = NN.flatten(), DD.flatten()
L_obs = true_loss(NN, DD) + np.random.normal(0, 0.008, size=NN.shape)   # + measurement noise

print(f'{len(NN)} synthetic training runs simulated')
print(f'N ranges {Ns.min():.0e} -> {Ns.max():.0e} params')
print(f'D ranges {Ds.min():.0e} -> {Ds.max():.0e} tokens')
print(f'observed loss ranges {L_obs.min():.3f} -> {L_obs.max():.3f}')

## Part 2: Fitting Power Laws From Scratch

Two from-scratch fits, in increasing order of ambition:

1. **Single-axis power law**: fit L(N) ≈ E + A/N^α using nothing but `np.polyfit` on log-transformed data — the textbook scaling-law technique.
2. **Naive two-axis fit**: estimate α by holding D at its *largest available value* and varying N (and vice-versa for β) — exactly the slicing strategy the original Kaplan et al. paper used.

We will see in Part 3 that step 2 is subtly biased, which sets up why a joint fit matters.

In [ ]:
def fit_power_law_axis(x, loss, E):
    """Fit loss = E + A / x^alpha via log-log linear regression.
    log(loss - E) = log(A) - alpha * log(x)  ->  straight line, slope = -alpha
    """
    y = np.log(np.clip(loss - E, 1e-8, None))
    slope, intercept = np.polyfit(np.log(x), y, 1)
    alpha = -slope
    A = np.exp(intercept)
    return A, alpha

# --- Naive single-axis slices: hold the OTHER variable at its largest available value ---
D_big = Ds.max()
mask_N = DD == D_big
A_scratch, alpha_scratch = fit_power_law_axis(NN[mask_N], L_obs[mask_N], E_true)

N_big = Ns.max()
mask_D = NN == N_big
B_scratch, beta_scratch = fit_power_law_axis(DD[mask_D], L_obs[mask_D], E_true)

print('=== Naive single-axis fit (Kaplan-style slicing) ===')
print(f'  alpha_scratch = {alpha_scratch:.3f}   (true alpha = {alpha_true})')
print(f'  beta_scratch  = {beta_scratch:.3f}   (true beta  = {beta_true})')
print(f'  A_scratch     = {A_scratch:.1f}     (true A     = {A_true})')
print(f'  B_scratch     = {B_scratch:.1f}     (true B     = {B_true})')

## Part 3: Verification — Naive Slicing vs. Joint Curve Fit

The naive estimates above are noticeably off, even though our "held-large" variable (D_big = 3×10¹¹ tokens) is the largest value we simulated. Why? Because B/D^β at D_big still contributes a non-trivial amount to the loss — it is *large*, but not large enough to be truly negligible. The slice we treated as "pure L(N)" secretly still has a hidden D-dependent offset baked in, which biases the fitted slope.

This is not a toy artifact — it is a real, documented subtlety in the scaling-laws literature, and one of the reasons later work moved to **jointly** fitting all the data to the full two-variable surface with a proper nonlinear optimizer, rather than fitting each axis in isolation.

In [ ]:
def surface(X, E, A, B, alpha, beta):
    N, D = X
    return E + A / N**alpha + B / D**beta

# Library: joint nonlinear least squares over ALL (N, D, loss) triples at once
popt, _ = curve_fit(
    surface, (NN, DD), L_obs,
    p0=[1.0, A_scratch, B_scratch, alpha_scratch, beta_scratch],
    maxfev=20000
)
E_fit, A_fit, B_fit, alpha_fit, beta_fit = popt

print('=== Comparison: naive slices vs. joint scipy.optimize.curve_fit vs. ground truth ===')
print(f'{"":12s}{"alpha":>10s}{"beta":>10s}{"A":>10s}{"B":>10s}{"E":>10s}')
print(f'{"naive":12s}{alpha_scratch:10.3f}{beta_scratch:10.3f}{A_scratch:10.1f}{B_scratch:10.1f}{"--":>10s}')
print(f'{"joint fit":12s}{alpha_fit:10.3f}{beta_fit:10.3f}{A_fit:10.1f}{B_fit:10.1f}{E_fit:10.3f}')
print(f'{"true":12s}{alpha_true:10.3f}{beta_true:10.3f}{A_true:10.1f}{B_true:10.1f}{E_true:10.3f}')
print()
print('✅ The joint fit recovers the true coefficients to within noise.')
print('⚠️  The naive single-axis fit is biased low on both exponents —')
print('   exactly the kind of methodological gap that separates Kaplan (2020) from Chinchilla (2022).')

## Part 4: Hyperparameter Experiments

Two experiments that turn the fitted surface into the actual decision scaling laws are used for: *given a compute budget, how big should the model be?*

1. **IsoFLOP curves** — for a few fixed compute budgets C, plot loss as a function of N (with D = C / 6N forced by the budget). Each curve has a minimum — that minimum *is* the compute-optimal model size for that budget.
2. **Where do real models sit?** — plot the optimal-N-vs-compute frontier implied by our fitted exponents, and place GPT-3, Gopher, and Chinchilla on it using their publicly reported parameter counts and token counts.

In [ ]:
def compute_optimal(C, E, A, B, alpha, beta):
    """Closed-form optimum of E + A/N^alpha + B/D^beta subject to D = C/(6N).
    Derivation: Hoffmann et al. 2022, Appendix.
    """
    a = beta / (alpha + beta)
    b = alpha / (alpha + beta)
    G = (alpha * A / (beta * B)) ** (1 / (alpha + beta))
    N_opt = G * (C / 6) ** a
    D_opt = (C / 6) / N_opt
    return N_opt, D_opt

# --- Experiment 1: IsoFLOP curves ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
compute_budgets = [1e19, 1e20, 1e21]
colors = sns.color_palette('viridis', len(compute_budgets))

for C, c in zip(compute_budgets, colors):
    N_opt, D_opt = compute_optimal(C, E_fit, A_fit, B_fit, alpha_fit, beta_fit)
    N_range = np.logspace(np.log10(N_opt) - 1.2, np.log10(N_opt) + 1.2, 300)
    D_range = (C / 6) / N_range
    L_range = surface((N_range, D_range), *popt)
    axes[0].plot(N_range, L_range, color=c, linewidth=2, label=f'C = {C:.0e} FLOPs')
    axes[0].scatter([N_opt], [surface((N_opt, D_opt), *popt)], color=c, s=80, zorder=5, edgecolor='black')

axes[0].set_xscale('log')
axes[0].set_xlabel('Model size N (parameters)'); axes[0].set_ylabel('Loss')
axes[0].set_title('IsoFLOP curves — minimum = compute-optimal N', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# --- Experiment 2: optimal frontier + real models ---
C_range = np.logspace(18, 24, 200)
N_opt_range = np.array([compute_optimal(C, E_fit, A_fit, B_fit, alpha_fit, beta_fit)[0] for C in C_range])
axes[1].plot(C_range, N_opt_range, color='#4C72B0', linewidth=2, label='Fitted compute-optimal frontier')

real_models = {
    'GPT-3 (175B, 300B tok)':   (175e9, 300e9),
    'Gopher (280B, 300B tok)':  (280e9, 300e9),
    'Chinchilla (70B, 1.4T tok)': (70e9, 1.4e12),
}
markers = ['o', 's', '^']
for (name, (N, D)), m in zip(real_models.items(), markers):
    C = 6 * N * D
    axes[1].scatter([C], [N], s=120, marker=m, label=name, zorder=5, edgecolor='black')

axes[1].set_xscale('log'); axes[1].set_yscale('log')
axes[1].set_xlabel('Training compute C (FLOPs)'); axes[1].set_ylabel('Model size N (parameters)')
axes[1].set_title('Real models vs. the compute-optimal frontier', fontweight='bold')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print('GPT-3 and Gopher sit ABOVE the fitted frontier for their compute budget — too many parameters,')
print('too little data. Chinchilla sits much closer to it — by design, that was the whole point of the paper.')

## Common Mistakes Students Make

**Mistake 1: Treating "scaling law" as "bigger is always better."**
A scaling law is a statement about the *optimal allocation of a fixed budget*, not a license to scale without limit. For a given compute budget C, there is one best (N, D) split — spending all of it on parameters (or all of it on data) is *worse* than the optimal split, not better.

**Mistake 2: Confusing compute-optimal with inference-optimal.**
The Chinchilla formula minimizes training loss for a fixed *training* compute budget. It says nothing about the cost of running the model afterward. A model that will be queried billions of times is often worth "overtraining" — making it smaller than compute-optimal and feeding it more tokens than the formula suggests — because a cheaper, slightly-less-loss-optimal model saves far more at inference time than it loses at training time. This is exactly why later models like LLaMA train smaller architectures on more tokens than naive Chinchilla-optimality would prescribe.

**Mistake 3: Estimating exponents from single-axis slices without checking the held variable is truly saturating.**
As Part 3 showed directly: holding D at the *largest value you happened to test* is not the same as holding D at a value where its contribution to loss is negligible. The bias this introduces is real, not just a synthetic artifact — it is part of why independent groups have derived different scaling exponents from similar data.

## Part 5: Interview Corner

**Question: A team has a fixed training budget of 1×10²² FLOPs. How would you decide between a 7B-parameter model and a 13B-parameter model, and how much data should each see?**

A complete answer:

**1. Don't guess — compute the optimal split.** Compute is approximately C ≈ 6ND. Given fitted (or published) exponents α, β and constants A, B for the model family, the optimal point is N_opt(C) = G·(C/6)^a, D_opt(C) = (C/6)/N_opt, with a = β/(α+β) and G = (αA/βB)^{1/(α+β)}.

**2. Compare both candidates against that optimum.** If N_opt(10²²) lands closer to 7B than 13B, the 7B model trained on its corresponding D_opt tokens will reach a *lower* training loss than the 13B model forced to use fewer tokens for the same compute — even though 13B is the "bigger" model.

**3. But then ask the question Chinchilla's formula doesn't answer: what happens after training?** If this model will be served millions of times in production, the smaller model is also cheaper per query. That pushes the decision *further* toward the smaller model and *more* tokens than even the compute-optimal point recommends — the inference-optimal point, not just the training-optimal one.

**The key insight:** scaling laws turn "how big should we make it?" from a guess into a calculation — but the calculation answers the training-cost question. A complete answer for a real product also accounts for the cost of inference, which the textbook formula deliberately leaves out.

## Key Takeaways

Five bullets you must remember for placement interviews:

- **Loss follows a power law in N, D, and C**: L ≈ E + A/N^α + B/D^β, a straight line on log-log axes across many orders of magnitude — surprisingly smooth for something as messy as deep learning.
- **E is an irreducible floor**: it represents the entropy of natural language itself — no amount of scale gets a model's loss below it.
- **Compute C ≈ 6ND links model size and data**: any fixed compute budget can be spent on a bigger model or more data, and there is one optimal split, not an arbitrary choice.
- **Chinchilla corrected Kaplan**: at fixed compute, parameters and data should scale at close to the *same* rate, not parameters dominating — which is why a 4x-smaller, 4x-more-data Chinchilla beat the larger Gopher.
- **Compute-optimal ≠ inference-optimal**: production systems often deliberately train smaller models on more tokens than the formula recommends, because inference cost recurs on every query while training cost is paid once.